# Model A: SmartPitch 기존 방식 (Baseline)

이 노트북은 세 전이확률 모델 비교의 기준점인 **Model A**를 정리한다.
Model A는 새로 학습하는 모델이 아니라, 기존 SmartPitch MDP의
**빈도 기반 전이확률 lookup**을 Baseline으로 재사용한다.

## 구성

1. MDP 배경 및 기존 방식 설명
2. Forward 방식의 한계
3. 세 모델 비교 개요
4. 구현 상태 및 Phase 6 계획

## 1. SmartPitch MDP 배경

SmartPitch는 야구의 **투수-타자 대결을 MDP로 모델링**하여 최적 구종/위치 전략을 제안하는 시스템이다.

### 1.1 MDP 구성 요소

| 요소 | 정의 | 예시 |
|------|------|------|
| 상태(State) | (볼카운트, 주자, 이닝, 아웃) | (2-1, 1루, 7회, 1아웃) |
| 행동(Action) | 구종 × 위치 | 4-seam Fastball, 높은 안쪽 |
| 전이확률 $P(s'\mid s, a)$ | 이 프로젝트의 핵심 추정 대상 | Ball → 3-1 카운트 확률 |
| 보상(Reward) | 기대 득점 변화 (RE24) | 홈런 허용 시 음수 보상 |

### 1.2 기존 전이확률 추정

기존 SmartPitch는 Statcast 데이터에서 **빈도를 직접 집계**한다:

$$\hat{P}(s' \mid s, a) = \frac{\text{count}(s \to s' \text{ via } a)}{\text{count}(s, a)}$$

구현은 단순하지만 **학습이 없는 non-parametric 방식**이다.

## 2. Forward 방식의 한계

### 2.1 실시간 투구 정보 미반영 (Stateless)

빈도 통계는 실제 투구 물리량(구속, 회전수, 무브먼트)을 입력으로 받지 않는다.
- 오늘 투수의 구속이 평소보다 5km/h 낮아도 같은 확률 반환
- 타자의 당일 타격 경향 반영 불가

### 2.2 희소 데이터 문제 (Sparse Counts)

세밀한 상태 조합(예: 3-2 카운트 + 만루 + 연장 11회)은 과거 데이터가 극히 적어
추정값의 분산이 크다. 심한 경우 count = 0으로 정의되지 않는다.

### 2.3 업데이트 불가 (Non-parametric)

시즌마다 전체 재집계가 필요하고, 온라인/점진적 학습이 불가능하다.

### 2.4 시퀀스 무시 (No Context)

앞선 투구들의 맥락(타자가 이전 3개 투구에서 계속 파울)을 전혀 활용하지 않는다.
이것이 Model C(Transformer, 400 pitch 시퀀스)의 핵심 개선점이다.

## 3. 세 모델 비교 개요

| | **Model A** (Baseline) | **Model B** (Otremba 2022) | **Model C** (MIT Sloan 2025) |
|---|---|---|---|
| 방식 | 빈도 lookup | MLP 지도학습 | Transformer 지도학습 |
| 입력 | 상태 조합 key | 77-dim 단일 투구 벡터 | 87-dim × 400 pitch 시퀀스 |
| 출력 | 전이확률 (lookup) | 4-class softmax | 24-dim (PR 10 + HL 9 + cont 5) |
| 파라미터 수 | 0 (non-parametric) | 27,012 | 9,659,672 |
| 시퀀스 맥락 | ✗ | ✗ | ✓ (400 pitches) |
| 실시간 투구 정보 | ✗ | ✓ | ✓ |
| 논문 | 내부 개발 | Otremba 2022 | MIT Sloan 2025 |
| 구현 상태 | 미구현 (wrapper 예정) | ✓ 완료 (val acc 60.6%) | 학습 진행 중 |

### Phase 6 비교 지표

| 지표 | 설명 |
|------|------|
| Cross-Entropy Loss | 낮을수록 좋음 |
| Brier Score | 확률 예측 MSE, 낮을수록 좋음 |
| Top-1 Accuracy | argmax 정확도 |
| Calibration (ECE) | 예측 확률과 실제 빈도의 일치도 |
| 추론 속도 | batch당 ms (CPU/CUDA) |

## 4. 구현 상태 및 계획

### 현재 상태 (Phase 5)

`src/models/model_a.py`는 아직 작성되지 않았다.
기존 SmartPitch 코드에서 전이확률 테이블을 가져와
`TransitionModel` 인터페이스의 wrapper로 감싸야 한다.

```python
# Phase 6에서 구현 예정: src/models/model_a.py
class SmartPitchBaseline(TransitionModel):
    """Wrapper around SmartPitch frequency-based transition probability table."""

    def __init__(self, lookup_table: dict):
        self.table = lookup_table

    def predict_proba(self, state_key) -> torch.Tensor:
        # 상태 키로 빈도 테이블 조회 → 확률 벡터 반환
        probs = self.table.get(state_key, self._uniform_prior())
        return torch.tensor(probs)
```

### Phase 6 로드맵

1. `src/models/model_a.py` — SmartPitch wrapper 구현
2. `src/evaluation/metrics.py` — 세 모델 공통 평가 함수
3. `src/evaluation/compare.py` — 나란히 비교 리포트
4. `notebooks/05_comparison.ipynb` — 시각화 및 결론

---

*다음 노트북: [03_model_b_otremba_2022.ipynb](03_model_b_otremba_2022.ipynb)*